# Materials + Made-In Country Comparison Notebook

This notebook uses **only your uploaded Mango and Zara CSVs**.

It focuses on:
- **materials**
- **made-in countries**
- **Men / Women / Kids**
- **predicted prices** using product attributes only
- creative charts for country and material comparisons


In [1]:
!pip -q install pandas numpy scikit-learn plotly openpyxl

In [2]:
from google.colab import files
uploaded = files.upload()

In [3]:
import os
import re
import numpy as np
import pandas as pd
from IPython.display import display
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def find_uploaded_file(keyword):
    candidates = [f for f in os.listdir('/content') if f.lower().endswith('.csv') and keyword.lower() in f.lower()]
    if not candidates:
        raise FileNotFoundError(f"Could not find a CSV with '{keyword}' in the filename inside /content.")
    candidates = sorted(candidates)
    return os.path.join('/content', candidates[0])

mango_path = find_uploaded_file('mango')
zara_path = find_uploaded_file('zara')

mango = pd.read_csv(mango_path)
zara = pd.read_csv(zara_path)

print("Mango file:", os.path.basename(mango_path), mango.shape)
print("Zara file :", os.path.basename(zara_path), zara.shape)

print("\nMango columns:")
print(mango.columns.tolist())

print("\nZara columns:")
print(zara.columns.tolist())

display(mango.head(2))
display(zara.head(2))

Mango file: mango_full_comparison_by_category.csv (74, 31)
Zara file : zara_combined_men_women_kids.csv (593, 21)

Mango columns:
['Product_ID', 'Segment', 'Source File', 'Specific_Category', 'US_url', 'IN_url', 'name', 'colour_from_title', 'US_price_current_usd', 'US_price_original_usd', 'US_discount_pct_display', 'US_discount_pct_calc', 'US_on_sale', 'IN_price_current_inr', 'IN_price_original_inr', 'IN_price_current_usd', 'IN_price_original_usd', 'IN_discount_pct_display', 'IN_discount_pct_calc', 'IN_on_sale', 'Savings_USD', 'materials_text', 'made_in', 'US_detail_selector', 'US_composition_selector', 'IN_detail_selector', 'IN_composition_selector', 'row_collected_at_local', 'row_collected_at_utc', 'fx_rate_live_usd_inr', 'fx_rate_timestamp_utc']

Zara columns:
['Product_ID', 'name', 'Category', 'colour_from_title', 'US_price_current_usd', 'US_price_original_usd', 'US_discount_pct_display', 'US_on_sale', 'IN_price_current_inr', 'IN_price_current_usd', 'IN_price_original_inr', 'IN_dis

,Product_ID,Segment,Source File,Specific_Category,US_url,IN_url,name,colour_from_title,US_price_current_usd,US_price_original_usd,...,materials_text,made_in,US_detail_selector,US_composition_selector,IN_detail_selector,IN_composition_selector,row_collected_at_local,row_collected_at_utc,fx_rate_live_usd_inr,fx_rate_timestamp_utc
0,17001208,Women,comparison_jeans,Women_jeans,https://shop.mango.com/us/en/p/women/jeans/wid...,https://shop.mango.com/in/en/p/women/jeans/wid...,High-rise wide leg jeans with pockets - Women ...,NaN,79.99,NaN,...,Composition: 100% cotton,Bangladesh\nDye,button:has-text('DETAILS'),NaN,text=DETAILS,NaN,2026-04-10T00:23:43.741040-04:00,2026-04-10T04:23:43.741062+00:00,92.45296,2026-04-10T04:22:00+00:00
1,17001212,Women,comparison_jeans,Women_jeans,https://shop.mango.com/us/en/p/women/jeans/wid...,https://shop.mango.com/in/en/p/women/jeans/wid...,Wide leg pleated jeans - Women | MANGO USA,NaN,69.99,NaN,...,Composition: 100% cotton | Lining: 65% polyest...,Bangladesh\nDye,text=DETAILS,text=Composition,text=DETAILS,NaN,2026-04-10T00:24:31.366829-04:00,2026-04-10T04:24:31.366850+00:00,92.47754,2026-04-10T04:23:00+00:00


,Product_ID,name,Category,colour_from_title,US_price_current_usd,US_price_original_usd,US_discount_pct_display,US_on_sale,IN_price_current_inr,IN_price_current_usd,...,IN_discount_pct_display,IN_discount_pct_calc,IN_on_sale,Savings_USD,materials_text,made_in,date,exchange_rate,Source File,Segment
0,7521601,LIMITED EDITION EMBROIDERED PUFFER JACKET - Be...,GENERAL,Beige,65.9,NaN,100.0,True,NaN,NaN,...,NaN,NaN,NaN,NaN,100% cotton | 100% polyester | 100% OCS certif...,India,2026-03-24,94.01,comparison_boynew,Kids
1,6858604,LIMITED EDITION EMBROIDERED FLORAL SLEEVELESS ...,GENERAL,Light blue,22.9,NaN,70.0,True,1650.0,17.55,...,-35.0,58.23,True,5.35,"Composition: 70% cotton, 30% linen | 70% cotto...",Portugal,2026-03-24,94.01,comparison_boynew,Kids


In [4]:
def clean_country(value):
    if pd.isna(value):
        return np.nan
    text = str(value).replace('\r', '\n').strip()
    if not text:
        return np.nan
    parts = [p.strip() for p in re.split(r'[\n|]+', text) if p.strip()]
    if not parts:
        return np.nan
    first = re.sub(r'[^A-Za-z& \-]', '', parts[0]).strip()
    if not first:
        return np.nan
    lower = first.lower()
    if 'designed in' in lower or lower in {'dye', 'wash', 'print'}:
        return np.nan
    return first.title()

def canonical_material(material_name):
    m = material_name.lower().strip()
    m = re.sub(r'[^a-z \-]', '', m)
    if 'cotton' in m:
        return 'cotton'
    if 'polyester' in m:
        return 'polyester'
    if 'linen' in m:
        return 'linen'
    if 'elastane' in m or 'spandex' in m:
        return 'elastane'
    if 'viscose' in m:
        return 'viscose'
    if 'wool' in m:
        return 'wool'
    if 'polyamide' in m or 'nylon' in m:
        return 'polyamide'
    if 'lyocell' in m:
        return 'lyocell'
    if 'modal' in m:
        return 'modal'
    if 'acrylic' in m:
        return 'acrylic'
    if 'silk' in m:
        return 'silk'
    if 'cashmere' in m:
        return 'cashmere'
    if 'leather' in m:
        return 'leather'
    if 'other fibres' in m or 'other fibers' in m:
        return 'other_fibres'
    return m

TRACKED_MATERIALS = [
    'cotton', 'polyester', 'linen', 'elastane', 'viscose', 'wool',
    'polyamide', 'lyocell', 'modal', 'acrylic', 'silk', 'cashmere',
    'leather', 'other_fibres'
]

def parse_materials(text):
    if pd.isna(text):
        return {}
    pieces = []
    seen = set()
    for raw_piece in str(text).split('|'):
        piece = raw_piece.strip()
        if not piece:
            continue
        piece = re.sub(r'(?i)composition:|lining:|material:|materials:', '', piece).strip()
        piece_key = piece.lower()
        if piece_key not in seen:
            seen.add(piece_key)
            pieces.append(piece)

    parsed = {}
    for piece in pieces:
        matches = re.findall(r'(\d+(?:\.\d+)?)%\s*([A-Za-z][A-Za-z \-]+)', piece, flags=re.I)
        for pct, mat in matches:
            pct = float(pct)
            key = canonical_material(mat)
            parsed[key] = parsed.get(key, 0.0) + pct
    return parsed

def build_base_frame(mango, zara):
    mango_std = pd.DataFrame({
        'Product_ID': mango['Product_ID'],
        'Brand': 'Mango',
        'Segment': mango['Segment'],
        'Category': mango['Specific_Category'].fillna(mango['Source File']),
        'Source_File': mango['Source File'],
        'name': mango['name'],
        'colour_from_title': mango['colour_from_title'],
        'US_price_current_usd': mango['US_price_current_usd'],
        'US_price_original_usd': mango['US_price_original_usd'],
        'IN_price_current_usd': mango['IN_price_current_usd'],
        'IN_price_original_usd': mango['IN_price_original_usd'],
        'US_on_sale': mango['US_on_sale'],
        'IN_on_sale': mango['IN_on_sale'],
        'Savings_USD': mango['Savings_USD'],
        'materials_text': mango['materials_text'],
        'made_in': mango['made_in']
    })

    zara_std = pd.DataFrame({
        'Product_ID': zara['Product_ID'],
        'Brand': 'Zara',
        'Segment': zara['Segment'],
        'Category': zara['Category'].fillna(zara['Source File']),
        'Source_File': zara['Source File'],
        'name': zara['name'],
        'colour_from_title': zara['colour_from_title'],
        'US_price_current_usd': zara['US_price_current_usd'],
        'US_price_original_usd': zara['US_price_original_usd'],
        'IN_price_current_usd': zara['IN_price_current_usd'],
        'IN_price_original_usd': np.nan,
        'US_on_sale': zara['US_on_sale'],
        'IN_on_sale': zara['IN_on_sale'],
        'Savings_USD': zara['Savings_USD'],
        'materials_text': zara['materials_text'],
        'made_in': zara['made_in']
    })

    df = pd.concat([mango_std, zara_std], ignore_index=True)
    df['made_in_country'] = df['made_in'].apply(clean_country)
    df['US_on_sale_num'] = df['US_on_sale'].fillna(False).astype(int)
    df['IN_on_sale_num'] = df['IN_on_sale'].fillna(False).astype(int)

    material_dicts = df['materials_text'].apply(parse_materials)
    for material in TRACKED_MATERIALS:
        df[f'mat_{material}'] = material_dicts.apply(lambda d: d.get(material, 0.0))

    df['material_points_total'] = df[[f'mat_{m}' for m in TRACKED_MATERIALS]].sum(axis=1)
    for material in TRACKED_MATERIALS:
        df[f'share_{material}'] = np.where(
            df['material_points_total'] > 0,
            df[f'mat_{material}'] / df['material_points_total'],
            0.0
        )

    df['material_variety'] = (df[[f'mat_{m}' for m in TRACKED_MATERIALS]] > 0).sum(axis=1)
    df['dominant_material'] = material_dicts.apply(lambda d: max(d, key=d.get) if len(d) > 0 else 'unknown')
    return df

df = build_base_frame(mango, zara)

print("Combined shape:", df.shape)
display(df.head(3))

print("\nTop made-in countries:")
display(df['made_in_country'].value_counts(dropna=False).head(15))

print("\nTop dominant materials:")
display(df['dominant_material'].value_counts().head(15))

Combined shape: (667, 50)


/tmp/ipykernel_707/1538876337.py:123: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['IN_on_sale_num'] = df['IN_on_sale'].fillna(False).astype(int)


,Product_ID,Brand,Segment,Category,Source_File,name,colour_from_title,US_price_current_usd,US_price_original_usd,IN_price_current_usd,...,share_polyamide,share_lyocell,share_modal,share_acrylic,share_silk,share_cashmere,share_leather,share_other_fibres,material_variety,dominant_material
0,17001208,Mango,Women,Women_jeans,comparison_jeans,High-rise wide leg jeans with pockets - Women ...,NaN,79.99,NaN,46.40,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,cotton
1,17001212,Mango,Women,Women_jeans,comparison_jeans,Wide leg pleated jeans - Women | MANGO USA,NaN,69.99,NaN,43.15,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,cotton
2,17007792,Mango,Women,Women_knitwear,comparison_knitwear,Medium-knit sweater - Women | MANGO USA,NaN,69.99,NaN,45.31,...,0.0,0.0,0.0,0.6,0.0,0.0,0.0,0.0,3,acrylic



Top made-in countries:


,count
made_in_country,
China,162
Morocco,130
Bangladesh,105
Turkey,99
Cambodia,43
NaN,38
Vietnam,20
Portugal,20
India,17



Top dominant materials:


,count
dominant_material,
cotton,320
polyester,222
unknown,48
linen,42
viscose,14
lyocell,6
wool,6
modal,4
silk,2


In [5]:
country_summary = (
    df.dropna(subset=['made_in_country'])
      .groupby(['Brand', 'Segment', 'made_in_country'], as_index=False)
      .agg(
          products=('Product_ID', 'count'),
          avg_us_price=('US_price_current_usd', 'mean'),
          avg_india_price=('IN_price_current_usd', 'mean'),
          avg_savings_usd=('Savings_USD', 'mean'),
          avg_material_variety=('material_variety', 'mean')
      )
      .sort_values(['products', 'avg_us_price'], ascending=[False, False])
)
country_summary = country_summary.round(2)

material_mix = (
    df.groupby(['Brand', 'Segment'], as_index=False)[[f'share_{m}' for m in TRACKED_MATERIALS]]
      .mean()
)
material_mix_pct = material_mix.copy()
for m in TRACKED_MATERIALS:
    material_mix_pct[f'share_{m}'] = (material_mix_pct[f'share_{m}'] * 100).round(2)

dominant_material_summary = (
    df.groupby(['Brand', 'Segment', 'dominant_material'], as_index=False)
      .agg(
          products=('Product_ID', 'count'),
          avg_us_price=('US_price_current_usd', 'mean'),
          avg_india_price=('IN_price_current_usd', 'mean'),
          avg_savings_usd=('Savings_USD', 'mean')
      )
      .sort_values(['products', 'avg_us_price'], ascending=[False, False])
      .round(2)
)

print("Country summary:")
display(country_summary.head(30))

print("\nMaterial mix by brand + segment:")
display(material_mix_pct)

print("\nDominant material summary:")
display(dominant_material_summary.head(30))

Country summary:


,Brand,Segment,made_in_country,products,avg_us_price,avg_india_price,avg_savings_usd,avg_material_variety
29,Zara,Men,China,81,102.97,73.60,44.73,1.36
33,Zara,Men,Morocco,68,136.16,83.83,52.33,1.88
16,Zara,Kids,Bangladesh,48,29.67,16.41,10.23,1.46
37,Zara,Men,Turkey,41,73.91,43.89,31.10,1.51
47,Zara,Women,Turkey,35,58.96,32.94,26.02,1.49
27,Zara,Men,Bangladesh,33,71.54,41.67,30.23,1.58
44,Zara,Women,Morocco,32,86.66,51.85,34.81,1.34
21,Zara,Kids,Morocco,25,48.62,29.46,16.74,1.56
18,Zara,Kids,China,25,47.98,24.46,11.69,1.04
41,Zara,Women,China,24,64.86,39.19,26.87,1.29



Material mix by brand + segment:


,Brand,Segment,share_cotton,share_polyester,share_linen,share_elastane,share_viscose,share_wool,share_polyamide,share_lyocell,share_modal,share_acrylic,share_silk,share_cashmere,share_leather,share_other_fibres
0,Mango,Kids,96.71,3.00,0.00,0.14,0.14,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,Mango,Men,40.10,20.81,5.35,0.20,1.28,23.12,0.46,0.02,0.00,1.99,0.00,2.50,4.17,0.00
2,Mango,Women,45.61,28.79,5.01,1.20,13.09,0.07,3.99,0.00,0.00,2.23,0.00,0.00,0.00,0.00
3,Zara,Kids,68.25,24.54,1.73,0.23,0.71,0.30,0.00,0.25,0.83,0.00,0.01,0.00,0.00,0.00
4,Zara,Men,39.69,35.17,8.97,0.14,2.78,1.78,0.15,1.43,0.53,0.04,0.80,0.01,0.00,0.02
5,Zara,Women,30.63,46.78,3.79,0.27,2.51,0.32,0.35,0.41,1.30,0.26,0.00,0.00,0.00,0.00



Dominant material summary:


,Brand,Segment,dominant_material,products,avg_us_price,avg_india_price,avg_savings_usd
16,Zara,Men,cotton,126,77.06,45.85,32.32
12,Zara,Kids,cotton,115,36.54,20.09,12.31
20,Zara,Men,polyester,106,114.15,83.57,49.15
29,Zara,Women,polyester,62,76.11,47.05,30.72
25,Zara,Women,cotton,38,61.56,36.00,26.79
14,Zara,Kids,polyester,38,47.69,26.84,13.75
17,Zara,Men,linen,31,121.73,73.89,47.84
22,Zara,Men,unknown,26,91.25,65.10,39.30
8,Mango,Women,cotton,20,76.39,51.05,25.34
30,Zara,Women,unknown,17,62.05,41.23,19.96


In [6]:
country_treemap = (
    df.dropna(subset=['made_in_country'])
      .groupby(['Brand', 'Segment', 'made_in_country'], as_index=False)
      .agg(products=('Product_ID', 'count'), avg_us_price=('US_price_current_usd', 'mean'))
)

fig1 = px.treemap(
    country_treemap,
    path=['Brand', 'Segment', 'made_in_country'],
    values='products',
    color='avg_us_price',
    title='Where the products are made: product count colored by average US price',
    hover_data={'avg_us_price': ':.2f', 'products': True}
)
fig1.show()

top_countries = df['made_in_country'].value_counts(dropna=True).head(10).index.tolist()
heat = (
    df[df['made_in_country'].isin(top_countries)]
      .pivot_table(index='made_in_country', columns='Segment', values='US_price_current_usd', aggfunc='median')
      .sort_index()
)

fig2 = px.imshow(
    heat,
    text_auto='.1f',
    aspect='auto',
    title='Median US price by made-in country and segment'
)
fig2.update_layout(xaxis_title='Segment', yaxis_title='Made in country')
fig2.show()

In [7]:
bubble = (
    df.dropna(subset=['made_in_country'])
      .groupby(['Brand', 'made_in_country'], as_index=False)
      .agg(
          products=('Product_ID', 'count'),
          avg_us_price=('US_price_current_usd', 'mean'),
          avg_savings=('Savings_USD', 'mean')
      )
)
bubble['avg_savings'] = bubble['avg_savings'].fillna(0)

fig3 = px.scatter(
    bubble,
    x='avg_us_price',
    y='avg_savings',
    size='products',
    color='Brand',
    hover_name='made_in_country',
    title='Average US price vs average US-India savings by made-in country'
)
fig3.update_layout(xaxis_title='Average US price (USD)', yaxis_title='Average savings (USD)')
fig3.show()

material_long = material_mix_pct.melt(
    id_vars=['Brand', 'Segment'],
    value_vars=[f'share_{m}' for m in TRACKED_MATERIALS],
    var_name='material',
    value_name='avg_share_pct'
)
material_long['material'] = material_long['material'].str.replace('share_', '', regex=False)

fig4 = px.bar(
    material_long,
    x='Segment',
    y='avg_share_pct',
    color='material',
    facet_col='Brand',
    barmode='stack',
    title='Average material mix by segment and brand'
)
fig4.update_layout(yaxis_title='Average share of parsed material composition (%)')
fig4.show()

In [8]:
top_materials = df['dominant_material'].value_counts().head(8).index.tolist()

country_material_heat = (
    df[df['dominant_material'].isin(top_materials) & df['made_in_country'].isin(top_countries)]
      .pivot_table(index='made_in_country', columns='dominant_material', values='US_price_current_usd', aggfunc='mean')
)

fig5 = px.imshow(
    country_material_heat,
    text_auto='.1f',
    aspect='auto',
    title='Average US price by made-in country and dominant material'
)
fig5.update_layout(xaxis_title='Dominant material', yaxis_title='Made in country')
fig5.show()

box_df = df[df['dominant_material'].isin(top_materials)].copy()

fig6 = px.box(
    box_df,
    x='dominant_material',
    y='US_price_current_usd',
    color='Brand',
    title='US price distribution by dominant material'
)
fig6.update_layout(xaxis_title='Dominant material', yaxis_title='US price (USD)')
fig6.show()

In [9]:
feature_columns = [
    'Brand', 'Segment', 'Category', 'made_in_country', 'dominant_material', 'colour_from_title',
    'US_on_sale_num', 'IN_on_sale_num', 'material_variety'
] + [f'share_{m}' for m in TRACKED_MATERIALS]

categorical_features = ['Brand', 'Segment', 'Category', 'made_in_country', 'dominant_material', 'colour_from_title']
numeric_features = ['US_on_sale_num', 'IN_on_sale_num', 'material_variety'] + [f'share_{m}' for m in TRACKED_MATERIALS]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value=0))
        ]), numeric_features)
    ]
)

def fit_price_model(input_df, target_col, label):
    model_df = input_df[feature_columns + [target_col]].dropna(subset=[target_col]).copy()
    if len(model_df) < 30:
        print(f"Not enough rows to model {label}.")
        return None, None, None

    X = model_df[feature_columns]
    y = model_df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    model = Pipeline([
        ('prep', preprocessor),
        ('rf', RandomForestRegressor(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=1,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    r2 = r2_score(y_test, preds)

    metrics_df = pd.DataFrame({
        'model': [label],
        'rows_used': [len(model_df)],
        'MAE': [round(mae, 3)],
        'RMSE': [round(rmse, 3)],
        'R2': [round(r2, 3)]
    })

    pred_df = X_test.copy()
    pred_df['actual_price'] = y_test.values
    pred_df['predicted_price'] = preds
    pred_df['abs_error'] = (pred_df['actual_price'] - pred_df['predicted_price']).abs()

    feature_names = model.named_steps['prep'].get_feature_names_out()
    importances = model.named_steps['rf'].feature_importances_
    importance_df = (
        pd.DataFrame({'feature': feature_names, 'importance': importances})
          .sort_values('importance', ascending=False)
          .reset_index(drop=True)
    )

    print(f"Metrics for {label}:")
    display(metrics_df)

    fig_pred = px.scatter(
        pred_df,
        x='actual_price',
        y='predicted_price',
        color='Segment',
        symbol='Brand',
        hover_data=['Category', 'made_in_country', 'dominant_material', 'abs_error'],
        title=f'Actual vs predicted price: {label}'
    )
    line_min = min(pred_df['actual_price'].min(), pred_df['predicted_price'].min())
    line_max = max(pred_df['actual_price'].max(), pred_df['predicted_price'].max())
    fig_pred.add_shape(type='line', x0=line_min, y0=line_min, x1=line_max, y1=line_max)
    fig_pred.update_layout(xaxis_title='Actual price', yaxis_title='Predicted price')
    fig_pred.show()

    error_summary = (
        pred_df.groupby(['Brand', 'Segment'], as_index=False)
               .agg(mean_abs_error=('abs_error', 'mean'))
               .round(2)
    )

    fig_err = px.bar(
        error_summary,
        x='Segment',
        y='mean_abs_error',
        color='Brand',
        barmode='group',
        title=f'Mean absolute error by brand and segment: {label}'
    )
    fig_err.update_layout(yaxis_title='Mean absolute error (USD)')
    fig_err.show()

    print(f"Top 20 features for {label}:")
    display(importance_df.head(20))

    return metrics_df, pred_df, importance_df

us_metrics, us_preds, us_importance = fit_price_model(df, 'US_price_current_usd', 'US current price model')
in_metrics, in_preds, in_importance = fit_price_model(df, 'IN_price_current_usd', 'India current price model')

Metrics for US current price model:


,model,rows_used,MAE,RMSE,R2
0,US current price model,667,23.43,43.124,0.462


Top 20 features for US current price model:


,feature,importance
0,cat__Segment_Kids,0.098442
1,num__share_wool,0.086454
2,cat__Segment_Men,0.083277
3,num__share_cotton,0.077139
4,num__share_polyester,0.074962
5,cat__made_in_country_Morocco,0.054337
6,cat__made_in_country_Pakistan,0.032546
7,num__IN_on_sale_num,0.027328
8,cat__colour_from_title_taupe brown,0.027114
9,num__material_variety,0.025107


Metrics for India current price model:


,model,rows_used,MAE,RMSE,R2
0,India current price model,540,21.787,41.051,0.044


Top 20 features for India current price model:


,feature,importance
0,num__share_cotton,0.144506
1,cat__Segment_Men,0.136103
2,num__share_polyester,0.129564
3,num__share_wool,0.065169
4,cat__colour_from_title_Black,0.055566
5,num__material_variety,0.040173
6,cat__Segment_Kids,0.030588
7,cat__made_in_country_Morocco,0.026825
8,num__share_viscose,0.025769
9,cat__colour_from_title_Gray green,0.018790


In [ ]:
country_summary.to_csv('/content/materials_country_country_summary.csv', index=False)
material_mix_pct.to_csv('/content/materials_country_material_mix_pct.csv', index=False)
dominant_material_summary.to_csv('/content/materials_country_dominant_material_summary.csv', index=False)

if us_preds is not None:
    us_preds.to_csv('/content/materials_country_us_price_predictions.csv', index=False)
if us_importance is not None:
    us_importance.to_csv('/content/materials_country_us_feature_importance.csv', index=False)
if in_preds is not None:
    in_preds.to_csv('/content/materials_country_india_price_predictions.csv', index=False)
if in_importance is not None:
    in_importance.to_csv('/content/materials_country_india_feature_importance.csv', index=False)

print("Saved these output files in /content:")
for f in sorted([x for x in os.listdir('/content') if x.startswith('materials_country_') and x.endswith('.csv')]):
    print(" -", f)